# Week 1 — Clean monthly PM2.5 (no hand merges)

Hey team — this notebook is step zero for honest modeling.

**What we use:** the file `df_model_monthly.csv` that we (or the earlier pipeline) already built. We are **not** re-joining OpenAQ, weather, or land cover here — that work is already inside this CSV.

**The one thing we fix:** some rows used **annual_fill** for PM2.5 (same yearly number copied into every month). That makes fake seasonality and can make models look better than they are. We **drop** those rows.

**What we keep:** only `openaq_monthly` — real measured PM2.5 for that month.

**Output:** `datasets/week1_openaq_base.csv` for Week 2.

**Heads-up on names:** we set `openaq_id` = `eoi_code`. That is the **EEA station code** (e.g. IT1975A), not OpenAQ’s internal numeric id. Week 2 needs a stable station id per row; this is it.


### Imports

Standard stuff: pandas for tables, pathlib for paths. If anything fails here, check you’re using the project conda env (e.g. `pm25`).


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
print("Libraries loaded")


### Paths

`MERGED_MONTHLY` is the big pre-merged table. `OUTPUT` is what we hand to Week 2. Keeping paths at the top means we only edit one place if files move.


In [ ]:
ROOT = Path(".")
MERGED_MONTHLY = ROOT / "df_model_monthly.csv"
OUTPUT = ROOT / "datasets" / "week1_openaq_base.csv"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("Merged input:", MERGED_MONTHLY)
print("Output:", OUTPUT)


### Load the merged table

Each row = one station (`eoi_code`) × one month. Peek at `data_source`: that column is the whole reason this notebook exists — we’ll filter on it next.


In [ ]:
df = pd.read_csv(MERGED_MONTHLY)
print("Shape (all rows):", df.shape)
print("Columns:", list(df.columns))
display(df.head())


### Count rows by `data_source`

Quick sanity check: you should see both `openaq_monthly` and `annual_fill`. Note the percentages — we expect to lose ~30% of rows when we drop annual fill, but the target becomes trustworthy.


In [ ]:
if "data_source" not in df.columns:
    raise ValueError("Expected column data_source. Use df_model_monthly.csv from this project.")

counts = df["data_source"].value_counts()
print(counts)
n = len(df)
print()
print("Real OpenAQ monthly %:", round(counts.get("openaq_monthly", 0) / n * 100, 2))
print("Annual fill %:", round(counts.get("annual_fill", 0) / n * 100, 2))


### Drop annual fill

After this, **every PM2_5 value is a real monthly observation**. We also drop `data_source` because it's always the same now.

If this cell makes the dataframe tiny, something's wrong with the input file.


In [ ]:
df = df[df["data_source"] == "openaq_monthly"].copy()
df = df.drop(columns=["data_source"])

print("Shape after filter:", df.shape)
display(df.head())


### Build `date` and `openaq_id`

`date` = first day of that month (easier for sorting and plots). `openaq_id` duplicates `eoi_code` so Week 2’s code can group by `openaq_id` without us rewriting all of Week 2.


In [ ]:
df["date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1))

if "eoi_code" not in df.columns:
    raise ValueError("Expected column eoi_code.")

df["openaq_id"] = df["eoi_code"].astype(str)

display(df[["openaq_id", "eoi_code", "Year", "Month", "date", "PM2_5"]].head())


### Column order

Just housekeeping: ids and target up front so the CSV is readable when someone opens it in Excel.


In [ ]:
front = [
    "openaq_id", "eoi_code", "Year", "Month", "date", "Season",
    "PM2_5", "PM10", "NO2", "O3",
    "Temp_Mean", "Wind_Speed", "Precipitation",
    "Altitude", "Latitude", "Longitude",
    "Station_Type", "Station_Area", "City",
    "Green_Ratio", "Population_Density",
]
cols = [c for c in front if c in df.columns]
rest = [c for c in df.columns if c not in cols]
df = df[cols + rest].copy()
print("Columns:", list(df.columns))


### Quality checks

Things we look for: no missing PM2.5, no duplicate station-month keys, and a sensible date range. Missing **other** columns (O3, PM10) is OK for now — Week 2/3 can impute or drop features; the target must be solid.


In [ ]:
key_cols = ["openaq_id", "Year", "Month"]
print("Rows:", len(df))
print("Missing PM2_5:", int(df["PM2_5"].isna().sum()))
print("Duplicate keys:", int(df.duplicated(subset=key_cols).sum()))
print("Date range:", df["date"].min(), "to", df["date"].max())
print()
print("Missing % (top 15):")
print((df.isna().mean() * 100).sort_values(ascending=False).head(15).round(2))


### Save

Write `week1_openaq_base.csv`. **Tell the team:** everyone should run this once (or after changing `df_model_monthly.csv`) before Week 2.


In [ ]:
df.to_csv(OUTPUT, index=False)
print("Saved:", OUTPUT, "| shape:", df.shape)
display(df.head())


### Next step

Open **`week2_features_and_validation.ipynb`** and point it at `datasets/week1_openaq_base.csv`.
